In [12]:
import torch
import torch.nn as nn
import sys
import numpy as np
from pathlib import Path
from src.model import MatrixFactorization


sys.path.append(str(Path.cwd().parent))
from src.metrics import evaluate
from src.data import load_ratings, time_split, build_id_maps

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

ratings = load_ratings()
train, test_warm = time_split(ratings)
user_to_idx, movie_to_idx = build_id_maps(train)
n_users, n_movies = len(user_to_idx), len(movie_to_idx)

print(f"n_users: {n_users}, n_movies: {n_movies}")

n_users: 5400, n_movies: 3662


In [ ]:
train_u = train["user_id"].map(user_to_idx).values
train_m = train["movie_id"].map(movie_to_idx).values
train_r = train["rating"].values

seen_by_user = {}
for u, m in zip(train_u, train_m):
    seen_by_user.setdefault(u, set()).add(m)

movie_counts = np.zeros(n_movies)
for m in train_m:
    movie_counts[m] += 1

movie_counts = movie_counts ** 0.75

movie_probs = movie_counts / movie_counts.sum()  
def sample_negative(user_idx: int) -> int:
    """Draw a popularity-weighted random movie this user has NOT interacted with."""
    seen = seen_by_user[user_idx]
    while True:
        random_movie = np.random.choice(n_movies, p=movie_probs)
        if random_movie not in seen:
            return random_movie

In [ ]:
def make_batch(batch_indices, alpha=40):
    """Given row indices into the training data, build a batch of
    positives and sampled negatives.

    Returns (users, movies, labels) as tensors.
    """
    users = train_u[batch_indices]     
    pos_movies = train_m[batch_indices]
    ratings = train_r[batch_indices]

    neg_movies = np.array([sample_negative(u) for u in users])

    batch_users = np.concatenate([users, users])  
    batch_movies = np.concatenate([pos_movies, neg_movies])
    batch_labels = np.concatenate([np.ones(len(users)), np.zeros(len(users))])
    batch_confidence = np.concatenate([1 + alpha * ratings, np.ones(len(users))])

    batch_users = torch.tensor(batch_users, dtype=torch.long)
    batch_movies = torch.tensor(batch_movies, dtype=torch.long)
    batch_labels = torch.tensor(batch_labels, dtype=torch.float)
    batch_confidence = torch.tensor(batch_confidence, dtype=torch.float)

    return batch_users, batch_movies, batch_labels, batch_confidence

In [32]:
bu, bm, bl, bc = make_batch(np.array([0, 1, 2]))
print(bu.shape, bm.shape, bl.shape, bc.shape)  # all (6,)
print(bc)  # first 3 are 1+40*rating, last 3 are 1.0

torch.Size([6]) torch.Size([6]) torch.Size([6]) torch.Size([6])
tensor([161., 161., 201.,   1.,   1.,   1.])


In [33]:
def wmf_loss(preds, labels, confidence):
    """Confidence-weighted squared error (WMF objective).

    Args:
        preds: model's raw dot-product outputs, shape (2B,)
        labels: 0/1 preference targets, shape (2B,)
        confidence: per-example confidence weights, shape (2B,)

    Returns:
        scalar loss
    """
    # 1. squared error per example: (labels - preds) squared
    squared_error = (labels - preds) ** 2

    # 2. weight each example's error by its confidence
    weighted = confidence * squared_error

    # 3. reduce to a single scalar (sum or mean — you decide
    return weighted.mean()

In [ ]:
model = MatrixFactorization(n_users, n_movies, k=50)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)

n_epochs = 5
batch_size = 1024
n_train = len(train_u)

for epoch in range(n_epochs):
    perm = np.random.permutation(n_train) 

    total_loss = 0.0
    for i in range(0, n_train, batch_size):
        batch_indices = perm[i:i + batch_size]

        users, movies, labels, confidence = make_batch(batch_indices)

        preds = model(users, movies)
        loss = wmf_loss(preds, labels, confidence)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_indices)

    avg_loss = total_loss / n_train
    print(f"Epoch {epoch+1}: WMF loss = {avg_loss:.4f}")

Epoch 1: WMF loss = 1313.0532
Epoch 2: WMF loss = 140.9984
Epoch 3: WMF loss = 37.5337
Epoch 4: WMF loss = 14.6006
Epoch 5: WMF loss = 8.0062


In [35]:
model.eval()
idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}
all_movie_idx = torch.arange(n_movies)

def recommend_mf(user_idx, k=10):
    with torch.no_grad():
        users = torch.full((n_movies,), user_idx, dtype=torch.long)
        scores = model(users, all_movie_idx)
    ranked = torch.argsort(scores, descending=True).tolist()
    seen = seen_by_user.get(user_idx, set())
    return [idx_to_movie[i] for i in ranked if i not in seen][:k]

warm_user_ids = list(test_warm["user_id"].unique())
recs_wmf = {uid: recommend_mf(user_to_idx[uid], k=10) for uid in warm_user_ids}

relevant_ratings = test_warm[test_warm["rating"] >= 4]
relevant_by_user = relevant_ratings.groupby("user_id")["movie_id"].apply(set).to_dict()

results_wmf = evaluate(recs_wmf, relevant_by_user, k=10)
print(f"Precision@10: {results_wmf['precision']:.4f}")
print(f"Recall@10:    {results_wmf['recall']:.4f}")
print(f"NDCG@10:      {results_wmf['ndcg']:.4f}")

Precision@10: 0.0023
Recall@10:    0.0003
NDCG@10:      0.0022


In [23]:
# pick one warm user, look at their top scores
uid = warm_user_ids[0]
user_idx = user_to_idx[uid]
with torch.no_grad():
    users = torch.full((n_movies,), user_idx, dtype=torch.long)
    scores = model(users, all_movie_idx)

print("Score range:", scores.min().item(), "to", scores.max().item())
print("Top recommended:", recommend_mf(user_idx, k=10))

Score range: -5.894778728485107 to 8.812064170837402
Top recommended: [np.int64(87), np.int64(2503), np.int64(2981), np.int64(3318), np.int64(131), np.int64(1822), np.int64(3596), np.int64(3048), np.int64(958), np.int64(3661)]


In [ ]:
for uid in warm_user_ids:
    if uid in relevant_by_user and len(relevant_by_user[uid]) > 0:
        print("User:", uid)
        print("Relevant:", sorted(relevant_by_user[uid])[:10])
        print("Recommended:", recs_wmf[uid])
        print("Overlap:", set(recs_wmf[uid]) & relevant_by_user[uid])
        break

User: 1875
Relevant: [10, 11, 260, 329, 539, 648, 674, 736, 750, 780]
Recommended: [np.int64(87), np.int64(2503), np.int64(2981), np.int64(3318), np.int64(131), np.int64(1822), np.int64(3596), np.int64(3048), np.int64(958), np.int64(3661)]
Overlap: set()


In [25]:
counts = train["movie_id"].value_counts()
print("Train counts for WMF recs:", [int(counts.get(m, 0)) for m in recs_wmf[1875]])

Train counts for WMF recs: [62, 9, 12, 30, 13, 17, 44, 62, 11, 70]


In [ ]:
uid = 1875
user_idx = user_to_idx[uid]

with torch.no_grad():
    users = torch.full((n_movies,), user_idx, dtype=torch.long)
    scores = model(users, all_movie_idx)

order = torch.argsort(scores, descending=True).tolist()
rank_of = {movie_idx: rank for rank, movie_idx in enumerate(order)}

liked_original = relevant_by_user[uid]
liked_dense = [movie_to_idx[m] for m in liked_original if m in movie_to_idx]
ranks = sorted(rank_of[d] for d in liked_dense)
print("Ranks of liked movies (out of", n_movies, "):", ranks[:20])

Ranks of liked movies (out of 3662 ): [468, 580, 699, 700, 955, 980, 1015, 1092, 1151, 1234, 1292, 1383, 1454, 1457, 1491, 1511, 1563, 1574, 1604, 1610]


In [37]:
# how big are the learned factor vectors vs the biases?
uf = model.user_factors.weight.detach()
mf = model.movie_factors.weight.detach()
ub = model.user_bias.weight.detach()
mb = model.movie_bias.weight.detach()

print("user factors  - mean abs:", uf.abs().mean().item())
print("movie factors - mean abs:", mf.abs().mean().item())
print("user bias     - mean abs:", ub.abs().mean().item())
print("movie bias    - mean abs:", mb.abs().mean().item())
print("global bias:", model.global_bias.item())

user factors  - mean abs: 0.4447106420993805
movie factors - mean abs: 0.35987362265586853
user bias     - mean abs: 0.11321372538805008
movie bias    - mean abs: 0.1823878139257431
global bias: 0.887752890586853


In [38]:
uid = 1875
user_idx = user_to_idx[uid]
with torch.no_grad():
    users = torch.full((n_movies,), user_idx, dtype=torch.long)
    scores = model(users, all_movie_idx)
order = torch.argsort(scores, descending=True).tolist()

seen = seen_by_user[user_idx]
top20 = order[:20]
print("Of top-20 scored movies, how many are TRAIN-seen:",
      sum(1 for m in top20 if m in seen), "/ 20")

Of top-20 scored movies, how many are TRAIN-seen: 0 / 20


In [42]:
u1 = user_to_idx[warm_user_ids[0]]
u2 = user_to_idx[warm_user_ids[30]]

recs1 = recommend_mf(u1, k=20)
recs2 = recommend_mf(u2, k=20)

print("User 1 top 20:", recs1)
print("User 2 top 20:", recs2)
print("Overlap:", len(set(recs1) & set(recs2)), "/ 20")

User 1 top 20: [np.int64(1442), np.int64(3570), np.int64(3289), np.int64(3514), np.int64(3790), np.int64(1404), np.int64(1749), np.int64(1581), np.int64(3433), np.int64(1329), np.int64(3437), np.int64(1906), np.int64(470), np.int64(973), np.int64(304), np.int64(2738), np.int64(2341), np.int64(2441), np.int64(1879), np.int64(985)]
User 2 top 20: [np.int64(2129), np.int64(1112), np.int64(3003), np.int64(1877), np.int64(3215), np.int64(3532), np.int64(2128), np.int64(3428), np.int64(2809), np.int64(3290), np.int64(1565), np.int64(2465), np.int64(3636), np.int64(3084), np.int64(3933), np.int64(3935), np.int64(2566), np.int64(2994), np.int64(2775), np.int64(3718)]
Overlap: 0 / 20


In [ ]:

counts = train["movie_id"].value_counts()
liked = relevant_by_user[1875]

popular_liked = [m for m in liked if counts.get(m, 0) > 200 and m in movie_to_idx]
niche_liked   = [m for m in liked if counts.get(m, 0) <= 200 and m in movie_to_idx]

pop_ranks   = sorted(rank_of[movie_to_idx[m]] for m in popular_liked)
niche_ranks = sorted(rank_of[movie_to_idx[m]] for m in niche_liked)

print("Popular liked movies, ranks:", pop_ranks[:15])
print("Niche liked movies, ranks:  ", niche_ranks[:15])

Popular liked movies, ranks: [700, 955, 980, 1015, 1092, 1234, 1292, 1383, 1454, 1457, 1491, 1511, 1563, 1574, 1604]
Niche liked movies, ranks:   [468, 580, 699, 1151, 2104, 2635, 3027]
